In [22]:
%pip install python-dotenv openai

  Using cached openai-3.24.0-py3-none-any.whl.metadata (44 kB)
  Using cached httpx2-2.13.1-py3-none-any.whl.metadata (9.8 kB)
  Using cached jiter-0.17.0-cp313-cp313-win_amd64.whl.metadata (5.3 kB)
  Using cached pydantic-2.13.5-py3-none-any.whl.metadata (110 kB)
  Using cached sniffio-1.3.1-py3-none-any.whl.metadata (3.9 kB)
  Using cached httpcore2-2.13.1-py3-none-any.whl.metadata (26 kB)
  Using cached truststore-0.10.4-py3-none-any.whl.metadata (4.4 kB)
  Using cached annotated_types-0.8.0-py3-none-any.whl.metadata (15 kB)
  Using cached pydantic_core-2.46.5-cp313-cp313-win_amd64.whl.metadata (6.7 kB)
  Using cached typing_inspection-0.4.4-py3-none-any.whl.metadata (2.6 kB)
Using cached openai-3.24.0-py3-none-any.whl (2.1 MB)
Using cached httpx2-2.13.1-py3-none-any.whl (95 kB)
Using cached httpcore2-2.13.1-py3-none-any.whl (83 kB)
Using cached jiter-0.17.0-cp313-cp313-win_amd64.whl (226 kB)
Using cached pydantic-2.13.5-py3-none-any.whl (472 kB)
Using cached pydantic_core-2.46.5-cp


[notice] A new release of pip is available: 25.1.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [5]:
import pandas as pd

df = pd.read_csv(
    "../data/processed/harmony_reviews_with_clusters.csv"
)

print(df.shape)
print(df["semantic_cluster_final"].value_counts())

(1604, 34)
semantic_cluster_final
0    1119
1     485
Name: count, dtype: int64


In [29]:
df.columns.tolist()

['id',
 'asins',
 'brand',
 'categories',
 'colors',
 'dateAdded',
 'dateUpdated',
 'dimension',
 'ean',
 'imageURLs',
 'keys',
 'manufacturer',
 'manufacturerNumber',
 'name',
 'primaryCategories',
 'reviews.date',
 'reviews.dateSeen',
 'reviews.doRecommend',
 'reviews.numHelpful',
 'reviews.rating',
 'reviews.sourceURLs',
 'reviews.text',
 'reviews.title',
 'reviews.username',
 'sourceURLs',
 'upc',
 'weight',
 'original_text',
 'cleaned_text',
 'cleaned_length',
 'cluster',
 'cluster_v3',
 'semantic_cluster',
 'semantic_cluster_final']

In [30]:
df["semantic_cluster_final"].value_counts().sort_index()

semantic_cluster_final
0    1119
1     485
Name: count, dtype: int64

In [6]:
#Prepare 10 reviews from each semantic cluster
cluster_inputs = {}

for cluster_num in sorted(df["semantic_cluster_final"].unique()):

    cluster_reviews = df[
        df["semantic_cluster_final"] == cluster_num
    ]["original_text"]

    reviews = cluster_reviews.sample(
        min(10, len(cluster_reviews)),
        random_state=42
    ).tolist()

    cluster_inputs[cluster_num] = reviews

print(cluster_inputs.keys())

dict_keys([np.int64(0), np.int64(1)])


In [7]:
for cluster_num, reviews in cluster_inputs.items():
    print(
        f"Cluster {cluster_num}: {len(reviews)} reviews"
    )

Cluster 0: 10 reviews
Cluster 1: 10 reviews


In [33]:
import os
from dotenv import load_dotenv

load_dotenv()

print("API key found:", bool(os.getenv("OPENAI_API_KEY")))

python-dotenv could not parse statement starting at line 1


API key found: True


In [34]:
import os
from getpass import getpass
from openai import OpenAI

os.environ["OPENAI_API_KEY"] = getpass("Enter your OpenAI API key: ")

client = OpenAI()

print("Client ready")

Client ready


In [35]:
response = client.responses.create(
    model="gpt-6-luna",
    input="Say hello in one short sentence."
)

print(response.output_text)

Hello!


In [36]:
def profile_cluster(reviews):

    reviews_text = "\n\n".join(
        f"Review {i+1}: {review}"
        for i, review in enumerate(reviews)
    )

    prompt = f"""
You are a customer review analyst.

Analyze these customer reviews for the Logitech Harmony Ultimate One remote.

Return:

1. Main Theme
2. Summary
3. Positive Aspects
4. Negative Aspects
5. Common Complaints
6. Business Insight

Rules:
- Use only evidence from the reviews.
- Do not invent information.
- Keep the answer concise.
- Separate product aspects from sentiment.

Reviews:

{reviews_text}
"""

    response = client.responses.create(
        model="gpt-6-luna",
        input=prompt
    )

    return response.output_text

In [37]:
cluster_profiles = {}

for cluster_num, reviews in cluster_inputs.items():

    print(f"\n========== CLUSTER {cluster_num} ==========")

    result = profile_cluster(reviews)

    cluster_profiles[cluster_num] = result

    print(result)


========== CLUSTER 0 ==========
1. **Main Theme**  
   A capable, easy-to-program universal remote that simplifies entertainment-system control, but has usability and reliability concerns.

2. **Summary**  
   Customers value the Harmony Ultimate One’s device support, activities, and touch screen. Some find its shape and button use awkward, and a few report problems with device-state consistency or worry about charging reliability.

3. **Positive Aspects**  
   - **Product aspects:** Controls multiple devices and activities; broad device database; responsive touch screen; helpful “Help” function; setup is described as easy, including importing settings from another Harmony remote.  
   - **Sentiment:** Several reviewers say it makes their systems easier to use and recommend or enjoy it.

4. **Negative Aspects**  
   - **Product aspects:** Some find it difficult to grip or use one-handed; its rounded base rocks when used on a table. One reviewer reports slow activity startup and about 

In [39]:
import json

# Convert NumPy cluster keys to normal Python integers/strings
cluster_profiles_clean = {
    str(int(cluster_num)): profile
    for cluster_num, profile in cluster_profiles.items()
}

with open(
    "../outputs/cluster_profiles.json",
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        cluster_profiles_clean,
        f,
        indent=4,
        ensure_ascii=False
    )

print("Cluster profiles saved successfully.")

Cluster profiles saved successfully.


In [10]:
from dotenv import load_dotenv
import os
from openai import OpenAI

load_dotenv(
    r"D:\PROJECTS\Customer_Review_Intelligence\.env"
)

api_key = os.getenv("OPENAI_API_KEY")

print("API key found:", bool(api_key))

client = OpenAI(api_key=api_key)

print("Client ready")

API key found: True
Client ready


In [11]:
import json

def generate_cluster_profile(cluster_num, reviews):

    reviews_text = "\n\n".join(
        f"Review {i+1}: {review}"
        for i, review in enumerate(reviews)
    )

    prompt = f"""
You are a business analyst analyzing customer reviews
for the Logitech Harmony Ultimate One remote.

These reviews belong to semantic Cluster {cluster_num}.

Analyze the reviews and return EXACTLY this structure:

Business Name: <2-3 word business-specific name>

Cluster Description:
<approximately 100 words explaining the common customer
need, experience, problems and themes in this cluster>

Cluster Summary:
<ONE sentence summarizing the cluster>

Rules:
- Base everything only on the supplied reviews.
- Do not invent information.
- The business name must describe the CUSTOMER/BUSINESS THEME,
  not simply say "Cluster 0" or "Positive Reviews".
- Keep the description close to 100 words.
- Separate the cluster topic from sentiment.
- Mention both positive and negative patterns when present.

Reviews:

{reviews_text}
"""

    response = client.responses.create(
        model="gpt-6-luna",
        input=prompt
    )

    return response.output_text.strip()

In [12]:
cluster_profiles_business = {}

for cluster_num, reviews in cluster_inputs.items():

    print(f"\n{'='*70}")
    print(f"CLUSTER {cluster_num}")
    print(f"{'='*70}")

    result = generate_cluster_profile(
        cluster_num,
        reviews
    )

    cluster_profiles_business[cluster_num] = result

    print(result)


CLUSTER 0
Business Name: Unified Entertainment Control

Cluster Description:
Customers use the Harmony Ultimate One to manage televisions, audio systems, players, cable or satellite equipment, and some smart-home devices with one remote. They value its broad device database, activity-based control, responsive touchscreen, helpful troubleshooting feature, and relatively straightforward setup, including importing settings from older Harmony models. Experiences are not uniformly smooth: some users report slow activity startup, devices ending up in unexpected states, and limited ways to correct problems. Others find the shape awkward to grip or use on a table, and mention short battery life or concerns about charging reliability. Overall, convenience and compatibility are balanced against usability and reliability drawbacks.

Cluster Summary:
Customers appreciate consolidating entertainment and smart-device controls in one programmable remote, while some encounter awkward ergonomics, acti